In [16]:
%env TALLTABLE_DB_DIR=/mnt/sdceph/users/spherex/talltable

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.cosmology import Planck18
from astropy import units as u
from astropy import constants as const
from astroquery.ipac.irsa import Irsa
from pprint import pprint

import healpy
from tqdm.auto import tqdm

from talltable import PixelQuery
import duckdb

env: TALLTABLE_DB_DIR=/mnt/sdceph/users/spherex/talltable


In [2]:
with fits.open('../data/LRG_NGC_clustering.dat.fits') as hdul:
    data = hdul[1].data

In [32]:
BITMASK_COL = 2**11 - 2**0
BITMASK_ROW = 2**23 - 2**12
BITMASK_DET = 2**27 - 2**24

rng = np.random.default_rng(119)

result = duckdb.sql(f"""
SELECT
    -- waveid,
    min(wavelength) AS min_lam, max(wavelength) AS max_lam,
    -- bandwidth,
    -- (waveid & {BITMASK_ROW}) >> 12 AS row,
    -- (waveid & {BITMASK_COL})       AS col,
    (waveid & {BITMASK_DET}) >> 24 AS det
FROM read_parquet('/mnt/sdceph/users/spherex/talltable/waves.parquet')
GROUP BY det
ORDER BY det
""").fetchnumpy()

wave_bands = {}
for i in range(len(result['det'])):
    _det = result['det'][i]
    _min = result['min_lam'][i]
    _max = result['max_lam'][i]
    
    r = rng.uniform(low=0.3, high=0.7)
    lam_c = (1 - r) * _min + r * _max
    wave_bands[_det] = (lam_c - 0.1, lam_c + 0.1)

    assert lam_c - 0.1 > _min
    assert lam_c + 0.1 < _max

    print(f'det {_det}: actual band ({_min:.3f}, {_max:.3f}), chosen band ({lam_c - 0.1:.3f}, {lam_c + 0.1:.3f})')

det 1: actual band (0.735, 1.127), chosen band (0.902, 1.102)
det 2: actual band (1.085, 1.668), chosen band (1.331, 1.531)
det 3: actual band (1.616, 2.440), chosen band (2.033, 2.233)
det 4: actual band (2.395, 3.867), chosen band (2.821, 3.021)
det 5: actual band (3.792, 4.439), chosen band (4.082, 4.282)
det 6: actual band (4.395, 5.019), chosen band (4.488, 4.688)


In [33]:
wave_bands

{np.int32(1): (np.float32(0.9021417), np.float32(1.1021417)),
 np.int32(2): (np.float32(1.3306594), np.float32(1.5306594)),
 np.int32(3): (np.float32(2.0329216), np.float32(2.2329214)),
 np.int32(4): (np.float32(2.8210704), np.float32(3.0210702)),
 np.int32(5): (np.float32(4.082157), np.float32(4.282157)),
 np.int32(6): (np.float32(4.488155), np.float32(4.6881547))}

In [47]:
rng = np.random.default_rng(121)

choices = rng.choice(len(data), size=(1,))

for choice in choices:
    ra = data['RA'][choice]
    dec = data['DEC'][choice]
    z = data['Z'][choice]

    table = PixelQuery(web=False).disc(ra, dec, 3).with_wavelengths().execute()
    lambs = table.column("wavelength").to_numpy()
    print(lambs.min(), lambs.max())

    query = PixelQuery(web=False).disc(ra, dec, 3).with_wavelengths()
    sql = query.sql()
    parts = query.partitions()
    print(parts)

    parts_sql = [
        f"'/mnt/sdceph/users/spherex/talltable/pixels/part={p}/compacted.parquet'"
        for p in parts
    ]
    if len(parts) > 1:
        parts_sql = f"[{','.join(parts_sql)}]"
    else:
        parts_sql = parts_sql[0]

    my_sql = f"""
WITH bins(bin_id, lo, hi) AS (
    VALUES
        (1, {wave_bands[1][0]:.3f}, {wave_bands[1][1]:.3f}),
        (2, {wave_bands[2][0]:.3f}, {wave_bands[2][1]:.3f}),
        (3, {wave_bands[3][0]:.3f}, {wave_bands[3][1]:.3f}),
        (4, {wave_bands[4][0]:.3f}, {wave_bands[4][1]:.3f}),
        (5, {wave_bands[5][0]:.3f}, {wave_bands[5][1]:.3f}),
        (6, {wave_bands[6][0]:.3f}, {wave_bands[6][1]:.3f})
)
SELECT
    b.bin_id, b.lo, b.hi,
    p.hphigh >> (2 * (22 - 15))                    AS hp15,
    weighted_avg(p.flux - p.zodi, 1. / p.variance) AS flux,
FROM bins b
JOIN read_parquet('/mnt/sdceph/users/spherex/talltable/waves.parquet') w
    ON w.wavelength BETWEEN b.lo AND b.hi
JOIN read_parquet({parts_sql}) p
    ON p.waveid = w.waveid
""" + '\n'.join(sql.split('\n')[12:]) + """
GROUP BY ALL
ORDER BY b.bin_id
"""

    print(my_sql)

    duckdb.sql(my_sql).show()

0.7364487 5.0074434
[1074839, 1074882, 1074888]

WITH bins(bin_id, lo, hi) AS (
    VALUES
        (1, 0.902, 1.102),
        (2, 1.331, 1.531),
        (3, 2.033, 2.233),
        (4, 2.821, 3.021),
        (5, 4.082, 4.282),
        (6, 4.488, 4.688)
)
SELECT
    b.bin_id, b.lo, b.hi,
    p.hphigh >> (2 * (22 - 15))                    AS hp15,
    weighted_avg(p.flux - p.zodi, 1. / p.variance) AS flux,
FROM bins b
JOIN read_parquet('/mnt/sdceph/users/spherex/talltable/waves.parquet') w
    ON w.wavelength BETWEEN b.lo AND b.hi
JOIN read_parquet(['/mnt/sdceph/users/spherex/talltable/pixels/part=1074839/compacted.parquet','/mnt/sdceph/users/spherex/talltable/pixels/part=1074882/compacted.parquet','/mnt/sdceph/users/spherex/talltable/pixels/part=1074888/compacted.parquet']) p
    ON p.waveid = w.waveid
WHERE
  -- disc(ra=189.90449140093455, dec=1.189696961819441, radius=0.05°), level-14 hphigh ranges
  (
    (p.hphigh BETWEEN 112802984689664 AND 112802984886271)
    OR (p.hphigh BETWEEN 

In [51]:
duckdb.sql(f"""
WITH bins(bin_id, lo, hi) AS (
    VALUES
        (1, {wave_bands[1][0]:.3f}, {wave_bands[1][1]:.3f}),
        (2, {wave_bands[2][0]:.3f}, {wave_bands[2][1]:.3f}),
        (3, {wave_bands[3][0]:.3f}, {wave_bands[3][1]:.3f}),
        (4, {wave_bands[4][0]:.3f}, {wave_bands[4][1]:.3f}),
        (5, {wave_bands[5][0]:.3f}, {wave_bands[5][1]:.3f}),
        (6, {wave_bands[6][0]:.3f}, {wave_bands[6][1]:.3f})
)
SELECT
    b.bin_id, b.lo, b.hi, count(w.wavelength)
FROM bins b
JOIN read_parquet('/mnt/sdceph/users/spherex/talltable/waves.parquet') w
    ON w.wavelength BETWEEN b.lo AND b.hi
GROUP BY ALL
ORDER BY b.bin_id
""")

┌────────┬──────────────┬──────────────┬─────────────────────┐
│ bin_id │      lo      │      hi      │ count(w.wavelength) │
│ int32  │ decimal(4,3) │ decimal(4,3) │        int64        │
├────────┼──────────────┼──────────────┼─────────────────────┤
│      1 │        0.902 │        1.102 │             2102300 │
│      2 │        1.331 │        1.531 │             1389821 │
│      3 │        2.033 │        2.233 │              958238 │
│      4 │        2.821 │        3.021 │              595397 │
│      5 │        4.082 │        4.282 │             1308214 │
│      6 │        4.488 │        4.688 │             1401670 │
└────────┴──────────────┴──────────────┴─────────────────────┘